# gguf

Pipeline-parallelism maximizes throughput, tensor-parallelism minimizes latency (requires fast GPU interconnect speeds).

to use tensor-parallelism: must manually specify context size

context size at Q4 weight + Q8 KV cache + 1 slot parallel:
- 31B: 20K
- 26B: 86K
- 12B: 100K

In [ ]:
%%bash
curl -LsSf https://llama.app/install.sh | sh
npm update -g npm

npm install -g localtunnel
pip install -qU huggingface_hub

# hf download --local-dir=/root/.llama-app mradermacher/gemma-4-31B-it-qat-q4_0-unquantized-uncensored-heretic-i1-GGUF gemma-4-31B-it-qat-q4_0-unquantized-uncensored-heretic.i1-Q4_K_M.gguf
hf download --local-dir=/root/.llama-app mradermacher/gemma-4-26B-A4B-it-qat-q4_0-unquantized-uncensored-heretic-v2-i1-GGUF gemma-4-26B-A4B-it-qat-q4_0-unquantized-uncensored-heretic-v2.i1-Q4_K_M.gguf
# hf download --local-dir=/root/.llama-app mradermacher/gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic-i1-GGUF gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic.i1-Q4_K_M.gguf

In [ ]:
import subprocess
f = open("stdout.txt", "w")
p = subprocess.Popen(["npx", "localtunnel", "--port", "11434"], bufsize=0, stdout=f, stderr=subprocess.STDOUT)

wait a few seconds

In [ ]:
!grep -F 'loca.lt' stdout.txt

go to the website above and follow instructions to unlock access

In [ ]:
!cd /root/.llama-app && GGML_CUDA_P2P=1 ./llama serve\
	--port 11434\
	--no-ui\
	--alias gemma4\
	--model gemma-4-26B-A4B-it-qat-q4_0-unquantized-uncensored-heretic-v2.i1-Q4_K_M.gguf\
	--n-gpu-layers all\
	--split-mode tensor\
	--ctx-size 84000\
	--load-mode none\
	--cache-ram 0\
	--no-cache-prompt\
	--no-cache-idle-slots\
	--parallel 1\
	--swa-full\
	--swa-checkpoints 0\
	--flash-attn on\
	--cache-type-k q8_0\
	--cache-type-v q8_0\
	--reasoning off\
	--temperature 0.3\
	--top-k 64\
	--top-p 0.95\
	--min-p 0.0\
	--repeat-penalty 1.1